# Aula 09 - EDA: Detec??o de fraudes
Kubeflow Pipelines | Kaggle Credit Card Fraud Detection

Execu??o dentro do cluster Kubernetes.

## 1. Instala??o do XGBoost

In [ ]:
%pip install --quiet xgboost==3.2.0

## 2. Bibliotecas e vers?es

In [ ]:

import sys
from pathlib import Path

sys.path.insert(0, '/home/jovyan/work')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb

sns.set_theme(style='whitegrid')

print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('xgboost:', xgb.__version__)


## 3. Carregamento do dataset

In [ ]:

arquivo = Path('/home/jovyan/data/creditcard.csv')

assert arquivo.exists(), 'CSV nao encontrado no PVC'

df = pd.read_csv(arquivo)

print('Shape:', df.shape)
print('Valores ausentes:', df.isna().sum().sum())

display(df.head())


## 4. Tipos e estat?sticas descritivas

In [ ]:

display(df.dtypes)
display(df.describe())


## 5. Desbalanceamento da vari?vel alvo

In [ ]:

class_counts = df['Class'].value_counts().sort_index()

print(class_counts)

for cls, n in class_counts.items():
    label = 'legitima' if cls == 0 else 'fraude'
    pct = 100 * n / len(df)
    print(f'{label}: {n} ({pct:.4f}%)')

fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    ['Legitima (0)', 'Fraude (1)'],
    class_counts.values,
    color=['steelblue', 'crimson']
)

ax.set_yscale('log')
ax.set_ylabel('Transacoes (escala log)')
ax.set_title('Distribuicao da target')

for i, v in enumerate(class_counts.values):
    ax.text(i, v, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 6. Distribui??o de Amount

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

amount = df['Amount'].to_numpy()

axes[0].hist(amount, bins=50, color='steelblue')
axes[0].set_title('Amount - escala linear')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequencia')

axes[1].hist(
    amount[amount > 0],
    bins=50,
    color='steelblue'
)

axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('Amount - escala log')
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

display(df['Amount'].describe())


## 7. Correla??o das vari?veis com fraude

In [ ]:

v_columns = [f'V{i}' for i in range(1, 29)]

correlations = df[v_columns + ['Amount']].corrwith(
    df['Class']
)

top_corr = correlations.abs().sort_values(
    ascending=False
).head(10)

top_corr_signed = correlations.loc[top_corr.index]

display(top_corr_signed.round(3))

fig, ax = plt.subplots(figsize=(9, 5))

colors = [
    'crimson' if v < 0 else 'steelblue'
    for v in top_corr_signed
]

ax.barh(
    top_corr_signed.index[::-1],
    top_corr_signed.values[::-1],
    color=colors[::-1]
)

ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Top 10 - Correlacao com Class')
ax.set_xlabel('Correlacao de Pearson')

plt.tight_layout()
plt.show()


## 8. Fraudes por quartil de Amount

In [ ]:

df_seg = df.assign(
    amount_quartil=pd.qcut(
        df['Amount'],
        q=4,
        labels=['Q1', 'Q2', 'Q3', 'Q4']
    )
)

fraud_por_quartil = (
    df_seg.groupby('amount_quartil', observed=True)['Class']
    .agg(['count', 'sum', 'mean'])
    .rename(columns={
        'count': 'n_transacoes',
        'sum': 'n_fraudes',
        'mean': 'fraud_rate'
    })
)

display(fraud_por_quartil)


## 9. Segmenta??o em low, medium e high

In [ ]:

df_seg = df.assign(
    amount_segment=pd.qcut(
        df['Amount'],
        q=[0, 0.25, 0.75, 1],
        labels=['low', 'medium', 'high']
    )
)

fraud_rate = (
    df_seg.groupby('amount_segment', observed=True)['Class']
    .agg(['count', 'sum', 'mean'])
    .rename(columns={
        'count': 'n_transacoes',
        'sum': 'n_fraudes',
        'mean': 'fraud_rate'
    })
)

display(fraud_rate)

assert fraud_rate['n_transacoes'].sum() == len(df)
assert fraud_rate['n_fraudes'].sum() == int(df['Class'].sum())


## 10. Estrat?gia de modelagem

A proposta da aula ? experimentar tr?s modelos XGBoost:

- low: transa??es de menor valor
- medium: transa??es intermedi?rias
- high: transa??es de maior valor

Na modelagem, os limites dos segmentos dever?o ser calculados somente com os dados de treinamento para evitar vazamento de informa??es.

Tamb?m ser? necess?rio comparar os tr?s modelos com uma abordagem de modelo ?nico e usar m?tricas adequadas ao desbalanceamento.

## 11. Verifica??o final

In [ ]:

assert df.shape == (284807, 31)
assert int(df['Class'].sum()) == 492
assert df.isna().sum().sum() == 0

print('EDA VALIDADA COM SUCESSO')
print('REGISTROS:', len(df))
print('FRAUDES:', int(df['Class'].sum()))
print('SEGMENTOS:', list(fraud_rate.index.astype(str)))
